# plot_06. Paper figure candidates

Every candidate figure of the planning notes of 9 October 2026: the panels A to F, the verification against ERA5, and the candidate layouts L1–L5 put together from them. Nothing here is the final choice. The point is to see every candidate on the real data and then pick one; notebook plot_05 keeps the figures already chosen.

Every candidate is a *block* drawn by one of this notebook's `draw_...(fig, ..., tag_start)` functions, into a whole figure or into one part of a layout (`fig.subfigures`), so the same block goes into several layouts: the maps layer by layer with `extant/plots/antarctica.py` (`draw_agreement_maps`, `draw_observed_maps`), the rest on axes from `extant/plots/charts.py` with its shared layers (lines, bands, a point per model and forcing) and the one-axis panels of `extant/plots/paper_panels.py` (`pp`: probability ratios, ridgelines, rank histograms, percentile tables). Every array a panel is given is made here, step by step, from what notebooks calc_02 and calc_03a to calc_03c saved (and notebook calc_04's sea ice): nothing is computed inside a figure.

| | Candidate | From |
|---|---|---|
| A1 | Mean change: the multi-model median with the AR6 hatching | 03 §4 `mean_bootstrap` |
| A2 | Mean change: the median, and how many models agree on its sign, per forcing | 03 §4 |
| A3 | Mean change, every model, per forcing | 03 §4 |
| A4 | Observed against historical: ERA5's trend, historical's, and where ERA5 falls among the members | calc_03a §9 `trend_maps`, calc_03c `detections` |
| A5 | Regional time series: each experiment's spread, and ERA5 | 03 §9 `regional_seasonal`, 02 §9 `era5_regional` |
| B1 | Width change: the median with the AR6 hatching, and the change in sea ice | 03 §2 `qrange_change`, 09 §3 |
| B2 | Width change, every model, per forcing | 03 §2 |
| B3 | The regional change at every percentile | 03 §9 `regional_quantile_curve` |
| C1 | Probability ratios of seasons beyond hist-nat's 5th and 95th percentiles, every model | 03 §10 `probability_ratio_regional` |
| C2 | The same, with the shift-only ratios | 03 §11 `probability_ratio_shift_only_regional` |
| C3 | Probability-ratio maps | 03 §10 `probability_ratio` |
| R1 | The distributions themselves, as ridgelines: every forcing, each region and season | 03 §9 `regional_seasonal` |
| R2 | The same for every model, historical | 03 §9 |
| D1–D4 | Sea ice and the width | 09 §3 (JASMIN), 03 §2, §9 |
| E1–E5 | The SAM: its forced change, and the mean and the width in each SAM phase | 03 *The SAM phases*, §2 and §4 by SAM phase |
| F1 | Regional synthesis: every forcing, the sum of the single forcings, and historical | 03 §6, §12 |
| F2 | Additivity: historical minus the sum of the single forcings | 03 §6, §12 |
| V1–V5 | Verification against ERA5 (Suarez-Gutierrez et al., 2021) | 02 §5–9 |
| S | Supplementary: how often each forcing is in each SAM phase, and every model's probability ratios | 03 *The SAM phases*, §10 |
| L1–L5 | The candidate layouts | all of the above |

**Conventions.** Rows are forcings and columns seasons, as in notebooks plot_03 to plot_05. A multi-model map is the median across the models, with the IPCC AR6 advanced approach to agreement (`config.SIGNIFICANT_THRESHOLD`, `config.SIGN_THRESHOLD`): no hatching where at least two-thirds of the models have a significant change and at least 80% of them agree on its sign (a robust change); diagonal lines where fewer than two-thirds are significant; crossed lines where they are but do not agree on the sign. Each model's significance is its own test against its own internal variability (notebook calc_02). The number of models is in each panel's corner. Print style (`fr.PAPER`), a page (183 mm) wide, with the caption as the markdown above each figure; every figure is saved as a PDF and a PNG in `figures/<variable>/candidates/`.

**On the sample** everything runs, but on 3×3 cells and two models, so the maps are dots, the SAM index is a made-up stand-in (notebook calc_02, *The SAM phases*), and the sea-ice panels (JASMIN) are left out.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from extant import config, datatree, era5_evaluation as ev, observed_change as oc, response_change as rc
from extant.config import ALPHA, FORCED_EXPERIMENTS, SIGNIFICANT_THRESHOLD, SIGN_THRESHOLD, WINDOW
from extant import significance as sig
from extant.logger import setup_logging
from extant.plots import antarctica, charts, era5_evaluation as evp, forced_response as fr, paper_panels as pp
from plotting_modules.constants import FORCING_COLORS, SEASONS
from extant.stats import fast_kde_xr, nan_quantile

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook; the candidates are
#(t): drawn in fr.PAPER's (STYLE)
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

The settings every notebook shares are in `extant/config.py` (`ALPHA`, `FORCED_EXPERIMENTS`, `SIGNIFICANT_THRESHOLD`, `SIGN_THRESHOLD`, `WINDOW`) and `plotting_modules/constants.py` (`SEASONS`). Only this notebook's own choices are here.

In [ ]:
VARIABLE = config.VARIABLES['tas']

#(t): The region of the regional panels, and the sea-ice region of D2-D4 (observed_change.REGIONS)
REGION = 'Antarctica (90–65°S)'
ICE_REGION = 'Southern Ocean (65–50°S)'

#(t): The probability ratios shown: of seasons beyond hist-nat's 95th (warm) or 5th (cold) percentile, which
#(t): hist-nat has a CHANCE of
CHANCE = 0.05

#(t): The first year of the time series (A5)
PLUME_START = 1950

#(t): The width of the smoothing kernel of the ridgelines (R1, R2), in °C
RIDGE_BANDWIDTH = 0.3

#(t): Print style, a page wide
STYLE = fr.PAPER
WIDTH = fr.PAGE_WIDTH

#(t): The maps of a figure alone: the room above them (inches; their column labels only, as there is no title),
#(t): and their size, four seasons filling the page's width
PAPER_TITLE_HEIGHT = 0.35
MAP_PANEL_SIZE = (WIDTH - antarctica.ROW_LABEL_WIDTH) / len(SEASONS)


In [ ]:
#(t): Where notebooks calc_03a to calc_03c, calc_02 and calc_04 saved their results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

In [ ]:
#(t): The colour scales: notebook plot_03's, and those of the probability ratio, the trends, ERA5's percentile, the change
#(t): in variance, and the share of seasons in a SAM phase
scales = fr.colour_scales(VARIABLE.name, alpha=ALPHA)
ratio_scale = pp.ratio_scale(f"Probability ratio, beyond hist-nat's {100 * CHANCE:g}th or "
                             f"{100 * (1 - CHANCE):g}th percentile")
trend_scale = fr.Scale(np.round(np.arange(-1, 1.01, 0.2), 2), 'RdBu_r', f'Trend ({VARIABLE.units} per decade)')
percentile_scale = pp.percentile_scale()
variance_scale = fr.Scale(np.round(np.arange(-2, 2.01, 0.4), 2), 'RdBu_r', f'Change in variance ({VARIABLE.units}²)')
PHASE_LABELS = {'negative': 'SAM negative', 'neutral': 'SAM neutral', 'positive': 'SAM positive'}

## Open the Saved Results

Run **one** of the next two sections. Both leave `RESULTS_DIR` and `ICE_RESULTS_DIR`, the folders notebooks calc_02's and calc_03a's to calc_03c's and notebook calc_04's results are opened from, and `CANDIDATE_DIR`, where the figures go (this repository's figures folder).

### Sample data

In [ ]:
#(t): The sample's results (notebooks calc_03a to calc_03c and calc_02's, and calc_04's sea ice), and where its figures go
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'sample'
CANDIDATE_DIR = Path('../figures') / VARIABLE.name / 'candidates_sample'
RESULTS_DIR, ICE_RESULTS_DIR, CANDIDATE_DIR

### Full data

In [ ]:
#(t): The full data's results, and where the candidates go
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'full'
CANDIDATE_DIR = Path('../figures') / VARIABLE.name / 'candidates'
RESULTS_DIR, ICE_RESULTS_DIR, CANDIDATE_DIR

### Saved results

In [ ]:
#(t): The hist-nat bootstrap of the mean change, with hist-nat's and every experiment's final-years mean
mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'mean_bootstrap.zarr').load()

In [ ]:
#(t): The width test: the change in Q95 − Q05, and its p-value
qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'qrange_change.zarr').load()

In [ ]:
#(t): Each experiment's change from its own 1850-1900 baseline
own_changes_da = xr.open_zarr(RESULTS_DIR / 'own_changes.zarr')['own_change'].load()

In [ ]:
#(t): Each experiment's change in variance from its own 1850-1900 baseline
own_variance_changes_da = xr.open_zarr(RESULTS_DIR / 'own_variance_changes.zarr')['own_variance_change'].load()

In [ ]:
#(t): The probability ratios
pr_ds = xr.open_zarr(RESULTS_DIR / 'probability_ratio.zarr').load()

In [ ]:
#(t): The probability ratios of the regional means
regional_pr_ds = xr.open_zarr(RESULTS_DIR / 'probability_ratio_regional.zarr').load()

In [ ]:
#(t): The shift-only ratios of the regional means
regional_shift_only_ds = xr.open_zarr(RESULTS_DIR / 'probability_ratio_shift_only_regional.zarr').load()

In [ ]:
#(t): The change in every 5th percentile, averaged over each region
regional_quantile_curve_da = xr.open_zarr(RESULTS_DIR / 'regional_quantile_curve.zarr')['regional_quantile_curve'].load()

In [ ]:
#(t): Every member's regional means, /<model>/<experiment>
regional_season_tree = xr.open_datatree(RESULTS_DIR / 'regional_seasonal.zarr', engine='zarr').load()

In [ ]:
#(t): The forced change of the SAM index
sam_change_da = xr.open_zarr(RESULTS_DIR / 'sam_change.zarr')['sam_change'].load()

In [ ]:
#(t): How often each experiment is in each SAM phase
sam_phase_share_da = xr.open_zarr(RESULTS_DIR / 'sam_phase_share.zarr')['phase_share'].load()

In [ ]:
#(t): The mean change in each SAM phase
sam_mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'sam_mean_bootstrap.zarr').load()

In [ ]:
#(t): The width test in each SAM phase
sam_qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'sam_qrange_change.zarr').load()

In [ ]:
#(t): ERA5's trend map and each model's ensemble-mean trend (notebook calc_03a)
trend_maps_ds = xr.open_zarr(RESULTS_DIR / 'trend_maps.zarr').load()

In [ ]:
#(t): The detection classes (notebook calc_03c)
detections_ds = xr.open_zarr(RESULTS_DIR / 'detections.zarr').load()

In [ ]:
#(t): ERA5's regional means
era5_regional_da = xr.open_zarr(RESULTS_DIR / 'era5_regional.zarr')[VARIABLE.name].load()

In [ ]:
#(t): The rank-histogram test of each region (notebook calc_03b)
regional_ranks_ds = xr.open_zarr(RESULTS_DIR / 'regional_ranks.zarr').load()

In [ ]:
#(t): ERA5's regional trend among each experiment's members (notebook calc_03a)
regional_consistency_ds = xr.open_zarr(RESULTS_DIR / 'regional_consistency.zarr').load()

In [ ]:
#(t): The spatial evaluation (notebook calc_03b)
spatial_table_ds = xr.open_zarr(RESULTS_DIR / 'spatial_table.zarr').load()

In [ ]:
#(t): The moment test at every grid point (notebook calc_03a)
eval_map_table_ds = xr.open_zarr(RESULTS_DIR / 'eval_map_table.zarr').load()

In [ ]:
#(t): The models: every count runs up to this many
N_MODELS = mean_bootstrap_ds.sizes['model']
list(mean_bootstrap_ds['model'].values)

### Sea ice

The change in sea-ice concentration (notebook calc_04, section 3), for B1 and D1–D4. Run **one** of the next two sections.

#### Without sea ice

In [ ]:
ice_change_da = None
ice_change_median_da = None
ice_change_median_da

#### With sea ice

In [ ]:
#(t): Each experiment's ensemble-mean concentration over its final years, on (model, experiment, season, lat, lon), %
ice_climatology_da = xr.open_zarr(ICE_RESULTS_DIR / 'climatology.zarr')['siconc'].load()

In [ ]:
#(t): The change: each experiment minus hist-nat
ice_change_da = ice_climatology_da - ice_climatology_da.sel(experiment='hist-nat')

In [ ]:
#(t): Its multi-model median, drawn as lines over the width maps (B1): ±5 and ±15%, dashed where ice is lost
ice_change_median_da = ice_change_da.median('model')

## Models Together

The AR6 advanced approach (`sig.model_agreement`) for each change: the median across models, how many of them have data, the share that agree on the sign of the median, and the three exclusive classes, `robust`, `conflicting` and `no_signal`. A model's change counts as significant by its own test in notebook calc_02: the hist-nat bootstrap for the mean and for the width, the bootstrap of members for a probability ratio.

In [ ]:
#(t): The mean
mean_agreement_ds = sig.model_agreement(mean_bootstrap_ds['mean_change'], mean_bootstrap_ds['pvalue'] < ALPHA,
                                        sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)

In [ ]:
#(t): The width, Q95 - Q05 of the internal variability
width_agreement_ds = sig.model_agreement(qrange_change_ds['qrange_change'], qrange_change_ds['qrange_pvalue'] < ALPHA,
                                         sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)

In [ ]:
#(t): The probability ratios as log2 (+1 doubles the chance; a ratio of 0 kept finite)
log_ratio_da = np.log2(pr_ds['pr'].sel(chance=CHANCE).clip(min=2.0 ** -10))

In [ ]:
#(t): and their agreement
ratio_agreement_ds = sig.model_agreement(log_ratio_da, pr_ds['pr_significant'].sel(chance=CHANCE),
                                         sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)

In [ ]:
#(t): The legend entries of the maps: AR6's hatching (diagonal lines where no robust change, crossed lines where the
#(t): models conflict), the dots of a single model's maps, and the sea-ice lines of B1. Labels broken onto lines to fit
no_signal_handle = Patch(facecolor='white', edgecolor='0.35', hatch='////',
                         label='No robust change (fewer than\n2/3 of the models significant)')
conflicting_handle = Patch(facecolor='white', edgecolor='0.35', hatch='xxxx',
                           label='Conflicting (2/3 significant, fewer\nthan 80% agree on the sign)')
not_significant_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none', label='Not significant')
ice_line_handle = plt.Line2D([], [], color='#333333', linewidth=1,
                             label='Change in sea-ice concentration\n(±5, ±15%; dashed: loss)')

In [ ]:
#(t): The agreement maps of A1, B1, C3 and E2, which also go into the layouts at the end
def draw_agreement_maps(fig, agreement_ds, row_dim, colours, row_labels=None, n_models_da=None, extra_handles=(),
                        tag_start=0):
    """The models' median change on (row_dim, season), hatched where they don't agree, in a figure or a layout's part.

    AR6's hatching: diagonal lines where no change is robust, crossed lines where the models conflict. A legend of it
    goes under the median's colour bar.

    Args:
        fig: A figure made with layout='constrained' (from map_figure_size with n_colorbar_rows=1 and n_legends=1),
            or one part of such a layout (fig.subfigures).
        agreement_ds: sig.model_agreement output on (row_dim, season, lat, lon), seasons in SEASONS' order: its
            median, no_signal and conflicting.
        row_dim: The rows' dim, e.g. 'experiment'.
        colours: The median's colours, e.g. scales.change.colours().
        row_labels: The rows' labels; their values if None.
        n_models_da: The number of models of each map, on (row_dim, season), written in its corner; none if None.
        extra_handles: More legend entries, for what the caller draws over the maps (B1's sea-ice lines).
        tag_start: The first map's letter (0 for a).

    Returns:
        The map axes, and the next map's letter (for a layout's next part).
    """
    rows = agreement_ds[row_dim].values
    axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)
    antarctica.heatmaps_of_antarctica(agreement_ds['median'], row_dim, 'season', axes=axes, cax=cax, **colours)
    antarctica.hatching_grid(agreement_ds['no_signal'], row_dim, 'season', axes=axes, pattern='////')
    antarctica.hatching_grid(agreement_ds['conflicting'], row_dim, 'season', axes=axes, pattern='xxxx')
    #(c): The number of models in each map's bottom-right corner
    if n_models_da is not None:
        for i, row in enumerate(rows):
            for j, season in enumerate(SEASONS):
                n_models = float(n_models_da.sel({row_dim: row, 'season': season}))
                axes[i, j].text(1.0, 0.0, f'n = {n_models:g}', transform=axes[i, j].transAxes, ha='right', va='bottom',
                                fontsize=STYLE.tick_size, color='0.3')
    #(c): The legend under the colour bar
    fig.legend(handles=[no_signal_handle, conflicting_handle, *extra_handles], loc='outside lower center',
               ncols=2 + len(extra_handles), frameon=False, fontsize=STYLE.cbar_label_size)

    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, rows if row_labels is None else row_labels, fontsize=STYLE.label_size)
    next_tag = antarctica.tag_panels(axes, start=tag_start, fontsize=STYLE.tag_size)
    return axes, next_tag

The chart blocks of the candidates, each drawn into a figure a page wide or into one part of a layout (`fig.subfigures`), as the agreement maps above: each makes its axes from a GridSpec (`fig.add_gridspec(...).subplots()`), draws its panels with the shared chart layers (or `pp`'s one-axis panels), labels them, letters them from `tag_start`, and returns its axes and the next letter. Each block gives its GridSpec the room round its panels in inches (`charts.margins`), and its legend goes in the room below (`loc='outside lower center'`). The layouts with maps are made with `layout='constrained'`, so that the maps' colour bars get their room under the maps (`antarctica.colorbar_axis_below`); there matplotlib sets the room round the panels itself, and makes room for the legend below them.

In [ ]:
#(t): A5: each experiment's median through time in one region (lines), the 5-95% of hist-nat and historical (bands),
#(t): and ERA5 (black dots, apart from historical's black line), one panel per season
def draw_plumes(fig, plume_da, era5_da, tag_start=0):
    """A5 in a figure (WIDTH wide) or one part of a layout.

    Args:
        plume_da: The pooled 5%, 50% and 95% of each experiment, on (experiment, quantile, year, season), as anomalies.
        era5_da: ERA5's anomalies on (year, season).
        tag_start: The first panel's letter (0 for a).

    Returns:
        The axes, and the next panel's letter.
    """
    #(c): The room round the panels in inches (charts.margins): above for the titles, below for the legend
    grid = fig.add_gridspec(1, len(SEASONS), wspace=0.12, **charts.margins(fig, top=PAPER_TITLE_HEIGHT, bottom=0.9))
    axes = grid.subplots(sharey=True, squeeze=False)
    shown_plume_da = plume_da.sel(year=slice(PLUME_START, None), season=SEASONS)
    #(c): Five overlapping bands could not be told apart: only hist-nat's and historical's
    for experiment in ['hist-nat', *FORCED_EXPERIMENTS]:
        if experiment in ('hist-nat', 'historical'):
            charts.quantile_bands_grid(shown_plume_da.sel(experiment=experiment), [(0.05, 0.95)], 'year', None,
                                       'season',
                                       axes=axes, color=FORCING_COLORS[experiment], alpha=0.2)
        charts.lines_grid(shown_plume_da.sel(experiment=experiment, quantile=0.5), 'year', None, 'season', axes=axes,
                          color=FORCING_COLORS[experiment], linewidth=1.0)
    charts.lines_grid(era5_da.sel(season=SEASONS), 'year', None, 'season', axes=axes,
                      **{**charts.ERA5_LINE, 'linestyle': 'none', 'markersize': 2.2})
    for ax, season in zip(axes[0], SEASONS):
        ax.axhline(0, color='0.6', linewidth=0.6, zorder=0)
        ax.set_title(season, loc='left')
    axes[0, 0].set_ylabel(f'Anomaly ({VARIABLE.units})')
    fig.legend(handles=charts.forcing_legend_handles(['hist-nat', *FORCED_EXPERIMENTS])
               + [Line2D([], [], **{**charts.ERA5_LINE, 'linestyle': 'none'}),
                  Patch(color='0.6', alpha=0.4, label='5–95%')],
               loc='outside lower center', ncols=len(FORCED_EXPERIMENTS) + 3, frameon=False)
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): B3, D3: the change at every percentile in each region (rows), the multi-model median (lines) and the models' range
#(t): (bands) of each forcing, seasons across
def draw_quantile_curves(fig, curves_da, regions, tag_start=0):
    """B3 or D3 in a figure (WIDTH wide) or one part of a layout.

    Args:
        curves_da: Every model's change at every percentile, on (model, experiment, quantile, region, season).
        regions: The regions, one row each, top first.
        tag_start: The first panel's letter (0 for a).

    Returns:
        The axes, and the next panel's letter.
    """
    #(c): The quantiles as percentiles (5 ... 95), for the x axis
    percentile_curves_da = curves_da.assign_coords(percentile=('quantile',
                                                               100 * curves_da['quantile'].values)).swap_dims(
        quantile='percentile').sel(region=regions, season=SEASONS)
    #(c): The room round the panels in inches (charts.margins): above for the titles, below for the x labels and the
    #(c): legend
    grid = fig.add_gridspec(len(regions), len(SEASONS), wspace=0.12, hspace=0.35,
                            **charts.margins(fig, top=PAPER_TITLE_HEIGHT, bottom=0.9))
    axes = grid.subplots(sharex=True, sharey='row', squeeze=False)
    for ax in axes.ravel():
        ax.axhline(0, color='0.6', linewidth=0.6, zorder=0)
    for experiment in FORCED_EXPERIMENTS:
        experiment_curves_da = percentile_curves_da.sel(experiment=experiment)
        charts.bands_grid(experiment_curves_da.min('model'), experiment_curves_da.max('model'), 'percentile', 'region',
                          'season', axes=axes, color=FORCING_COLORS[experiment], alpha=0.15)
        charts.lines_grid(experiment_curves_da.median('model'), 'percentile', 'region', 'season', axes=axes,
                          color=FORCING_COLORS[experiment], linewidth=1.4)
    #(c): The x axis is shared: its range and ticks set once
    axes[0, 0].set_xlim(0, 100)
    axes[0, 0].set_xticks([0, 50, 100])
    for ax, season in zip(axes[0], SEASONS):
        ax.set_title(season, loc='left')
    for ax, region in zip(axes[:, 0], regions):
        ax.set_ylabel(f'{region}\nΔ ({VARIABLE.units})')
    for ax in axes[-1]:
        ax.set_xlabel('Percentile')
    fig.legend(handles=charts.forcing_legend_handles(FORCED_EXPERIMENTS)
               + [Patch(color='0.6', alpha=0.3, label='Range of the models')], loc='outside lower center',
               ncols=len(FORCED_EXPERIMENTS) + 1, frameon=False)
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): C1, C2: the probability ratios of a warm (top) and a cold season (bottom), forcings along x, every model with its
#(t): bootstrap range and the multi-model median, and with C2 the shift-only ratio (open markers)
def draw_probability_ratios(fig, ratios_ds, shift_only=False, tag_start=0):
    """C1 or C2 in a figure (WIDTH wide) or one part of a layout.

    Args:
        ratios_ds: The region's ratios, pr, pr_lower, pr_upper (and pr_shift_only for C2) on (model, experiment, tail,
            season).
        shift_only: Draw the shift-only ratios too (C2).
        tag_start: The first panel's letter (0 for a).

    Returns:
        The axes, and the next panel's letter.
    """
    #(c): The room round the panels in inches (charts.margins): above for the titles, below for the turned x labels and
    #(c): the legend
    grid = fig.add_gridspec(2, len(SEASONS), wspace=0.12, hspace=0.2,
                            **charts.margins(fig, top=PAPER_TITLE_HEIGHT, bottom=1.3))
    axes = grid.subplots(sharex=True, sharey=True, squeeze=False)
    pp.probability_ratios_grid(ratios_ds.sel(tail=['warm', 'cold'], experiment=FORCED_EXPERIMENTS, season=SEASONS),
                               'tail',
                               'season', axes=axes, shift_only=shift_only)
    for ax, season in zip(axes[0], SEASONS):
        ax.set_title(season, loc='left')
    for ax, tail in zip(axes[:, 0], ['warm', 'cold']):
        ax.set_ylabel(f'{tail}\nProbability ratio')
    #(c): The forcings' names turned, so four fit under a narrow panel
    for ax in axes[-1]:
        for label in ax.get_xticklabels():
            label.set(rotation=40, ha='right')
    ratio_handles = charts.model_legend_handles(ratios_ds['model'].values) + [
        Patch(color='0.6', alpha=0.5, label='Multi-model median')]
    if shift_only:
        ratio_handles.append(Line2D([], [], linestyle='none', marker='o', markersize=4, markerfacecolor='white',
                                    markeredgecolor='0.35', label='Shift of the mean only'))
    fig.legend(handles=ratio_handles, loc='outside lower center', ncols=len(ratio_handles), frameon=False)
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): R1, R2: whole distributions as overlapping ridges, one per forcing or model (top to bottom), coloured by their
#(t): mean, one panel per season; dotted, hist-nat's 5th and 95th percentiles
def draw_ridgelines(fig, densities_da, means_da, ridge_dim, levels_da, tag_start=0, left_margin=charts.LEFT_MARGIN):
    """R1 or R2 in a figure (WIDTH wide) or one part of a layout.

    Args:
        densities_da: The densities on (ridge_dim, season, x), the ridges top first.
        means_da: Their means on (ridge_dim, season): each ridge's colour and number.
        ridge_dim: 'experiment' (R1) or 'model' (R2).
        levels_da: The dotted lines' values on (season, quantile).
        tag_start: The first panel's letter (0 for a).
        left_margin: Room for the ridges' names (inches).

    Returns:
        The axes, and the next panel's letter.
    """
    ridges = list(densities_da[ridge_dim].values)
    #(c): One height per unit of density in every panel, the highest ridge reaching 1.8 baselines up, so the areas compare
    height_scale = 1.8 / float(densities_da.sel(season=SEASONS).max())
    #(c): The room round the panels in inches (charts.margins): above for the titles, left for the ridges' names (the
    #(c): colour bar
    #(c): takes its room from the panels)
    grid = fig.add_gridspec(1, len(SEASONS), wspace=0.08,
                            **charts.margins(fig, left=left_margin, top=PAPER_TITLE_HEIGHT, bottom=0.3))
    axes = grid.subplots(sharex=True, sharey=True, squeeze=False)
    for ax, season in zip(axes[0], SEASONS):
        pp.plot_ridgelines(densities_da.sel(season=season), means_da.sel(season=season), ridge_dim, height_scale,
                           limit=3.0, lines=levels_da.sel(season=season), ax=ax)
        ax.set_title(season, loc='left')
        ax.set_xlabel(f'Anomaly from hist-nat ({VARIABLE.units})')
    #(c): The y axis is shared: the ridges' names once, ridge k from the top on baseline n - 1 - k, and room for the
    #(c): highest ridge, 1.8 baselines up
    axes[0, 0].set_yticks(np.arange(len(ridges))[::-1], ridges)
    axes[0, 0].set_ylim(-0.15, len(ridges) - 1 + 1.8 + 0.15)
    fig.colorbar(ScalarMappable(norm=Normalize(-3.0, 3.0), cmap=pp.RIDGE_CMAP), ax=axes, location='bottom', shrink=0.6,
                 pad=0.3, extend='both', label=f'Mean change ({VARIABLE.units})')
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): D2, D4, E3, E5: one change against another, a point per model (marker) and forcing (colour), one panel per season
def draw_scatter(fig, x_da, y_da, xlabel, ylabel, tag_start=0):
    """D2, D4, E3 or E5 in a figure (WIDTH wide) or one part of a layout.

    Args:
        x_da, y_da: The two changes on (model, experiment, season).
        xlabel, ylabel: What they are, with their units.
        tag_start: The first panel's letter (0 for a).

    Returns:
        The axes, and the next panel's letter.
    """
    #(c): The room round the panels in inches (charts.margins): above for the titles, then the panels and a row for the
    #(c): two
    #(c): legends
    grid = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.45,
                            **charts.margins(fig, top=PAPER_TITLE_HEIGHT, bottom=0.05))
    axes = grid[0].subgridspec(1, len(SEASONS), wspace=0.12).subplots(sharex=True, sharey=True, squeeze=False)
    legend_axes = grid[1].subgridspec(2, 1).subplots()
    for legend_ax in legend_axes:
        legend_ax.axis('off')
    for ax, season in zip(axes[0], SEASONS):
        charts.plot_scatter_by_forcing(x_da.sel(experiment=FORCED_EXPERIMENTS, season=season),
                                       y_da.sel(experiment=FORCED_EXPERIMENTS, season=season), ax=ax)
        ax.set_title(season, loc='left')
        ax.set_xlabel(xlabel)
    axes[0, 0].set_ylabel(ylabel)
    legend_axes[0].legend(handles=charts.forcing_legend_handles(FORCED_EXPERIMENTS), loc='center',
                          ncols=len(FORCED_EXPERIMENTS),
                          frameon=False)
    legend_axes[1].legend(handles=charts.model_legend_handles(x_da['model'].values), loc='center',
                          ncols=x_da.sizes['model'], frameon=False)
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): E1, E4, F1, S1: a change for each forcing (bars: the multi-model median; markers: each model), one row per
#(t): quantity, seasons across
def draw_forcing_bars(fig, rows, x_dim='experiment', bars=FORCED_EXPERIMENTS, tag_start=0):
    """E1, E4, F1 or S1 in a figure (WIDTH wide) or one part of a layout.

    Args:
        rows: Each row's y label -> its values on (model, x_dim, season), top first.
        x_dim: The dim along x, 'experiment', or 'term' for the additivity (with 'sum of parts').
        bars: Its values, in order.
        tag_start: The first panel's letter (0 for a).

    Returns:
        The axes, and the next panel's letter.
    """
    #(c): The room round the panels in inches (charts.margins): above for the titles, below for the turned x labels and
    #(c): the legend
    grid = fig.add_gridspec(len(rows), len(SEASONS), wspace=0.12, hspace=0.2,
                            **charts.margins(fig, top=PAPER_TITLE_HEIGHT, bottom=1.3))
    axes = grid.subplots(sharex=True, sharey='row', squeeze=False)
    for row_axes, (label, values_da) in zip(axes, rows.items()):
        charts.model_dots_grid(values_da.sel({x_dim: bars, 'season': SEASONS}), x_dim, None, 'season', axes=row_axes,
                               color={**FORCING_COLORS, **pp.OTHER_COLORS})
        row_axes[0].set_ylabel(label)
    for ax, season in zip(axes[0], SEASONS):
        ax.set_title(season, loc='left')
    #(c): The bars' names turned, so they fit under a narrow panel
    for ax in axes[-1]:
        for tick_label in ax.get_xticklabels():
            tick_label.set(rotation=40, ha='right')
    models = next(iter(rows.values()))['model'].values
    fig.legend(handles=charts.model_legend_handles(models)
               + [Patch(color='0.6', alpha=0.5, label='Multi-model median')], loc='outside lower center',
               ncols=len(models) + 1, frameon=False)
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): V1: ERA5's regional rank histograms against the range a perfect model gives, every model (rows) and season
def draw_rank_histograms(fig, ranks_ds, tag_start=0):
    """V1 in a figure (WIDTH wide) or one part of a layout.

    Args:
        ranks_ds: One region, experiment and treatment of regional_ranks: obs_frequency, pseudo_lower, pseudo_upper on
            (model, season, rank), and percentile, verdict on (model, season, statistic).
        tag_start: The first panel's letter (0 for a).

    Returns:
        The axes, and the next panel's letter.
    """
    models = list(ranks_ds['model'].values)
    #(c): The room round the panels in inches (charts.margins): above for the titles, left for the models' names, below
    #(c): for the
    #(c): x labels and the legend
    grid = fig.add_gridspec(len(models), len(SEASONS), wspace=0.08, hspace=0.25,
                            **charts.margins(fig, left=1.1, top=PAPER_TITLE_HEIGHT, bottom=0.9))
    axes = grid.subplots(sharex=True, sharey=True, squeeze=False)
    for i, model in enumerate(models):
        for j, season in enumerate(SEASONS):
            pp.plot_regional_rank_histogram(ranks_ds.sel(model=model, season=season), ax=axes[i, j])
        axes[i, 0].set_ylabel(f'{model}\n%')
    for ax, season in zip(axes[0], SEASONS):
        ax.set_title(season, loc='left')
    for ax in axes[-1]:
        ax.set_xlabel('Rank (scaled)')
    fig.legend(handles=pp.rank_histogram_legend_handles(), loc='outside lower center', ncols=3, frameon=False)
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): V4: ERA5's percentile among each experiment's members (columns), every model (rows), one table per season,
#(t): outlined where flagged
def draw_percentile_tables(fig, percentile_da, verdict_da, experiments, tag_start=0):
    """V4 in a figure (WIDTH wide) or one part of a layout.

    Args:
        percentile_da, verdict_da: On (model, experiment, season).
        experiments: The columns of each table, in order.
        tag_start: The first panel's letter (0 for a).

    Returns:
        The axes, and the next panel's letter.
    """
    #(c): The room round the panels in inches (charts.margins): above for the titles, left for the models' names (the
    #(c): colour bar
    #(c): takes its room from the tables)
    grid = fig.add_gridspec(1, len(SEASONS), wspace=0.05,
                            **charts.margins(fig, left=1.1, top=PAPER_TITLE_HEIGHT, bottom=0.3))
    axes = grid.subplots(sharey=True, squeeze=False)
    for ax, season in zip(axes[0], SEASONS):
        table_image = pp.plot_percentile_table(percentile_da.sel(season=season, experiment=experiments),
                                               verdict_da.sel(season=season, experiment=experiments), ax=ax)
        ax.set_title(season, loc='left')
    fig.colorbar(table_image, ax=axes, location='bottom', shrink=0.6, pad=0.25,
                 ticks=evp.PERCENTILE_BOUNDS,
                 label=pp.percentile_scale().label)
    return axes, charts.tag_panels(axes, start=tag_start)

In [ ]:
#(t): The number of models in each panel
mean_n_models_da = mean_agreement_ds['n_models'].max(['lat', 'lon'])
width_n_models_da = width_agreement_ds['n_models'].max(['lat', 'lon'])
ratio_n_models_da = ratio_agreement_ds['n_models'].max(['lat', 'lon'])
mean_n_models_da.to_pandas()

In [ ]:
#(t): How much of the area south of 60°S each forcing robustly changes (%)
robust_area_df = pd.DataFrame({
    'mean': (100 * sig.area_mean(mean_agreement_ds['robust'].sel(lat=slice(None, -60)))).to_series(),
    'width': (100 * sig.area_mean(width_agreement_ds['robust'].sel(lat=slice(None, -60)))).to_series(),
}).unstack('season').round(0)
robust_area_df

## A. Mean

**A1. The forced change in the mean.** Multi-model median change in the seasonal mean temperature, 1993–2013, against hist-nat over the same years, for each forcing (rows) and season (columns). Hatching after IPCC AR6: diagonal lines, fewer than two-thirds of the models have a significant change (hist-nat bootstrap, p < 0.05); crossed lines, at least two-thirds do but fewer than 80% agree on its sign; none, a robust change. n: the number of models.

In [ ]:
#(t): A1: the models' median change in the mean, every forcing (rows) and season, hatched where they don't agree
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                        n_colorbar_rows=1, n_legends=1,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes, next_tag = draw_agreement_maps(fig, mean_agreement_ds.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS),
                                         'experiment',
                                         scales.change.colours(), n_models_da=mean_n_models_da)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'A1_mean_agreement')

**A2. The mean change, and how many models agree on its sign.** For each forcing (one figure each): the multi-model median change (top) and the number of models whose change has the sign of the median (bottom).

In [ ]:
#(t): How many models agree on the sign of the median
mean_sign_count_da = (mean_agreement_ds['sign_agreement'] * mean_agreement_ds['n_models']).round()
mean_sign_count_da

In [ ]:
#(t): A2, one figure per forcing: the median (top) and how many models agree on its sign (bottom)
count_scale = fr.majority_scale(N_MODELS, 'Models agreeing on the sign of the median')
a2_figures = {}
for experiment in FORCED_EXPERIMENTS:
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(SEASONS), n_colorbar_rows=2,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(SEASONS))
        #(c): Each row's colour bar under it
        median_cax = antarctica.colorbar_axis_below(axes[0])
        count_cax = antarctica.colorbar_axis_below(axes[1])
        antarctica.heatmaps_of_antarctica(mean_agreement_ds['median'].sel(experiment=experiment, season=SEASONS), None, 'season',
                                          axes=axes[0], cax=median_cax, **scales.change.colours())
        antarctica.heatmaps_of_antarctica(mean_sign_count_da.sel(experiment=experiment, season=SEASONS), None, 'season',
                                          axes=axes[1], cax=count_cax, **count_scale.colours())
        antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
        antarctica.label_rows(axes, [f'{experiment}\nmedian', 'Models\nagreeing'], fontsize=STYLE.label_size)
        antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
    a2_figures[experiment] = fig

In [ ]:
for experiment, fig in a2_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'A2_mean_and_sign_{experiment}')

**A3. The mean change in every model.** For each forcing (one figure each): every model's change (rows) in each season. Dots: not significant in that model (hist-nat bootstrap, p ≥ 0.05).

In [ ]:
#(t): Not significant in a model, where it has data
mean_not_significant_da = ~(mean_bootstrap_ds['pvalue'] < ALPHA) & mean_bootstrap_ds['mean_change'].notnull()
mean_not_significant_da

In [ ]:
#(t): A3, one figure per forcing: every model (rows), dotted where not significant
a3_figures = {}
models = mean_bootstrap_ds['mean_change']['model'].values
for experiment in FORCED_EXPERIMENTS:
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                            n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
        cax = antarctica.colorbar_axis_below(axes)
        antarctica.heatmaps_of_antarctica(mean_bootstrap_ds['mean_change'].sel(experiment=experiment, season=SEASONS), 'model', 'season', axes=axes,
                                          cax=cax, **scales.change.colours())
        antarctica.not_significant_dots_grid(mean_not_significant_da.sel(experiment=experiment, season=SEASONS), 'model', 'season',
                                             axes=axes)
        antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
        antarctica.label_rows(axes, models, fontsize=STYLE.label_size)
        antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
        fig.legend(handles=[not_significant_handle], loc='outside lower center', fontsize=STYLE.cbar_label_size)
    a3_figures[experiment] = fig

In [ ]:
for experiment, fig in a3_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'A3_mean_every_model_{experiment}')

**A4. The observed change against historical.** (a–d) ERA5's trend over the years ERA5 and the models share (1979–2013) and (e–h) the multi-model median of historical's ensemble-mean trend over the same years. (i–l) ERA5's percentile among historical's members (each model's 1979–2013 trends, the median across models): central colours, ERA5 behaves like a member; dark blue or red, ERA5's trend is below or above 95% of the members. Dots: ERA5's trend is outside hist-nat's 5–95% range in at least two-thirds of the models (detected).

In [ ]:
#(t): ERA5's trend and historical's multi-model median trend, along a 'source' dim
observed_trend_da = xr.concat([trend_maps_ds['era5_trend'],
                               trend_maps_ds['model_trend'].sel(experiment='historical', drop=True).median('model')],
                              dim=xr.Variable('source', ['ERA5', 'historical (median)']))
observed_trend_da

In [ ]:
#(t): ERA5's percentile among historical's members, the median across models
era5_percentile_da = detections_ds['historical_percentile'].median('model')
era5_percentile_da

In [ ]:
#(t): Where ERA5's trend is outside hist-nat's range (detected: classes 1 and 2) in at least two-thirds of the models
n_detection_models_da = detections_ds['detection_class'].notnull().sum('model')
n_detected_da = detections_ds['detection_class'].isin([1, 2]).sum('model')
detected_da = (n_detected_da >= SIGNIFICANT_THRESHOLD * n_detection_models_da) & (n_detection_models_da > 0)
detected_da

In [ ]:
#(t): A4's maps, which also go into L4: ERA5's trend and historical's median trend (one colour bar), then ERA5's
#(t): percentile among historical's members, dotted where ERA5's trend is detected
def draw_observed_maps(fig, tag_start=0):
    """A4's maps in a figure made with layout='constrained' (from map_figure_size with 3 rows, n_colorbar_rows=2
    and n_legends=1) or one part of such a layout.

    Returns:
        The map axes, and the next map's letter.
    """
    axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
    #(c): The trends' colour bar under rows 1 and 2, the percentile's under row 3
    trend_cax = antarctica.colorbar_axis_below(axes[:2])
    percentile_cax = antarctica.colorbar_axis_below(axes[2])
    antarctica.heatmaps_of_antarctica(observed_trend_da.sel(season=SEASONS), 'source', 'season', axes=axes[:2],
                                      cax=trend_cax, **trend_scale.colours())
    antarctica.heatmaps_of_antarctica(era5_percentile_da.sel(season=SEASONS), None, 'season', axes=axes[2],
                                      cax=percentile_cax, **percentile_scale.colours())
    antarctica.hatching_grid(detected_da.sel(season=SEASONS), None, 'season', axes=axes[2], pattern='....')
    #(c): The legend under the colour bars
    detected_handle = Patch(facecolor='white', edgecolor='0.35', hatch='....',
                            label="Detected: ERA5 outside hist-nat's 5–95% in at least 2/3 of the models")
    fig.legend(handles=[detected_handle], loc='outside lower center', frameon=False, fontsize=STYLE.cbar_label_size)

    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, [*observed_trend_da['source'].values, "ERA5's percentile"], fontsize=STYLE.label_size)
    next_tag = antarctica.tag_panels(axes, start=tag_start, fontsize=STYLE.tag_size)
    return axes, next_tag

In [ ]:
#(t): A4 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes, next_tag = draw_observed_maps(fig)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'A4_observed_vs_historical')

**A5. Regional time series.** Seasonal mean temperature averaged over the region, as anomalies from historical's 1979–2013 mean in each model (ERA5: from its own 1979–2013 mean). Bands: the 5–95% range of every member of every model, pooled, for each experiment; lines: their medians; black: ERA5.

In [ ]:
#(t): Every member's regional means (notebook calc_02, section 9) as one Dataset, members padded with NaN where models differ
regional_season_ds = datatree.tree_to_dataset(regional_season_tree, dims=['model', 'experiment'])

In [ ]:
#(t): Each model's historical ensemble mean over 1979-2013: the baseline of its anomalies
historical_baseline_da = (regional_season_ds[VARIABLE.name].sel(experiment='historical', year=slice(1979, 2013))
                          .mean(['member', 'year']))

In [ ]:
#(t): Every member's anomaly from it, every experiment
regional_anomaly_da = regional_season_ds[VARIABLE.name] - historical_baseline_da

In [ ]:
#(t): Every member of every model pooled
pooled_anomaly_da = regional_anomaly_da.stack(pooled=['model', 'member'])

In [ ]:
#(t): The pooled 5%, 50% and 95% of each experiment in each year
plume_da = nan_quantile(pooled_anomaly_da, [0.05, 0.5, 0.95], 'pooled')
plume_da

In [ ]:
#(t): ERA5's anomalies from its own 1979-2013 mean
era5_anomaly_da = era5_regional_da - era5_regional_da.sel(year=slice(1979, 2013)).mean('year')
era5_anomaly_da

In [ ]:
#(t): A5 alone, a page wide
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 2.6))
    axes, _ = draw_plumes(fig, plume_da.sel(region=REGION), era5_anomaly_da.sel(region=REGION))

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'A5_regional_time_series')

## B. Width

**B1. The forced change in width.** Multi-model median change in the width Q95 − Q05 of the internal variability (each member minus its ensemble's forced response), 1993–2013 against hist-nat over the same years. Hatching as in A1 (hist-nat bootstrap, p < 0.05). Lines (with the sea ice, D1): the multi-model median change in sea-ice concentration, ±5 and ±15%, dashed where ice is lost.

In [ ]:
#(t): B1: the models' median change in the width, every forcing (rows) and season, hatched where they don't agree, with
#(t): the median change in sea ice as lines (if the sea ice was opened)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                        n_colorbar_rows=1, n_legends=1,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes, next_tag = draw_agreement_maps(fig, width_agreement_ds.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS),
                                         'experiment',
                                         scales.spread_change.colours(), n_models_da=width_n_models_da,
                                         extra_handles=[] if ice_change_median_da is None else [ice_line_handle])
    if ice_change_median_da is not None:
        antarctica.contour_lines_grid(ice_change_median_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS),
                                      (-15, -5, 5, 15), 'experiment', 'season', axes=axes)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'B1_width_agreement')

**B2. The change in width in every model.** As A3, for the width. Dots: not significant (hist-nat bootstrap, p ≥ 0.05).

In [ ]:
width_not_significant_da = ~(qrange_change_ds['qrange_pvalue'] < ALPHA) & qrange_change_ds['qrange_change'].notnull()
width_not_significant_da

In [ ]:
#(t): B2, one figure per forcing: every model (rows), dotted where not significant
b2_figures = {}
models = qrange_change_ds['qrange_change']['model'].values
for experiment in FORCED_EXPERIMENTS:
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                            n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
        cax = antarctica.colorbar_axis_below(axes)
        antarctica.heatmaps_of_antarctica(qrange_change_ds['qrange_change'].sel(experiment=experiment, season=SEASONS), 'model', 'season', axes=axes,
                                          cax=cax, **scales.spread_change.colours())
        antarctica.not_significant_dots_grid(width_not_significant_da.sel(experiment=experiment, season=SEASONS), 'model', 'season',
                                             axes=axes)
        antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
        antarctica.label_rows(axes, models, fontsize=STYLE.label_size)
        antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
        fig.legend(handles=[not_significant_handle], loc='outside lower center', fontsize=STYLE.cbar_label_size)
    b2_figures[experiment] = fig

In [ ]:
for experiment, fig in b2_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'B2_width_every_model_{experiment}')

**B3. The change at every percentile.** The change in every 5th percentile, 1993–2013 against hist-nat over the same years, averaged over each region (rows), for each forcing: the multi-model median (lines) and the models' range (shading). A flat line is a shift of the whole distribution; a line falling to the right, the cold tail warming more than the warm tail (a narrowing).

In [ ]:
#(t): B3 alone: every region (rows)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 1.7 * len(oc.REGIONS) + 1.3))
    axes, _ = draw_quantile_curves(fig, regional_quantile_curve_da, list(oc.REGIONS))

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'B3_quantile_curves')

## C. Extremes: Probability Ratios

**C1. How much more likely a warm or a cold season.** Probability ratio of a season warmer than hist-nat's 95th percentile (top) or colder than its 5th percentile (bottom), regional means, 1993–2013, each forcing against hist-nat. Markers: each model (shape), with its 95% bootstrap range (whole members resampled); bars: the multi-model median. Log scale; a ratio of 0 sits on the bottom edge.

In [ ]:
#(t): The region's ratios
regional_ratio_ds = regional_pr_ds.sel(region=REGION, chance=CHANCE)
regional_ratio_ds[['pr', 'pr_lower', 'pr_upper']].to_dataframe().unstack('season').round(2)

In [ ]:
#(t): C1 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 4.2))
    axes, _ = draw_probability_ratios(fig, regional_ratio_ds)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'C1_probability_ratios')

**C2. The shift of the mean alone, and the rest.** As C1, with open markers for the shift-only ratio, the ratio a shift of the mean alone would give (hist-nat's seasons moved by the change in the mean). The gap between a filled and an open marker is what the change in width and shape adds; it matters most for the cold tail.

In [ ]:
#(t): The region's shift-only ratios (notebook calc_02, section 11) next to them
regional_both_ratios_ds = xr.merge([regional_ratio_ds, regional_shift_only_ds.sel(region=REGION, chance=CHANCE)],
                                   compat='no_conflicts')
regional_both_ratios_ds[['pr', 'pr_shift_only']].to_dataframe().unstack('season').round(2)

In [ ]:
#(t): C2 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 4.2))
    axes, _ = draw_probability_ratios(fig, regional_both_ratios_ds, shift_only=True)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'C2_probability_ratios_shift_only')

**C3. Probability-ratio maps.** Multi-model median probability ratio of a season warmer than hist-nat's 95th percentile (one figure) or colder than its 5th (the other), 1993–2013 against hist-nat. Orange: more likely; purple: less likely. Hatching as in A1, with significance from the bootstrap (the 95% range of a model's ratio excludes 1).

In [ ]:
#(t): C3, one figure per tail: the models' median probability ratio, every forcing (rows) and season, hatched where they
#(t): don't agree
c3_figures = {}
for tail in ('warm', 'cold'):
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                            n_colorbar_rows=1, n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes, next_tag = draw_agreement_maps(fig, ratio_agreement_ds.sel(tail=tail, experiment=FORCED_EXPERIMENTS,
                                                                         season=SEASONS),
                                             'experiment', ratio_scale.colours(), n_models_da=ratio_n_models_da.sel(tail=tail))
    c3_figures[tail] = fig

In [ ]:
for tail, fig in c3_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'C3_probability_ratio_maps_{tail}')

## R. Distributions: Ridgelines

Every number in A–C summarises a distribution; these show the distributions. Each ridge is the distribution of one experiment's seasonal means over 1993–2013, as anomalies from its model's hist-nat mean over the same years, so hist-nat is centred on 0 in every model. Ridges are coloured by their mean, blue through purple to red, with the mean written beside them. The dotted lines are hist-nat's 5th and 95th percentiles, so the share of a ridge beyond them is what the probability ratios (C1, C2) count: a ridge that moves right puts more beyond the warm line, and one that also narrows puts less beyond the cold line than the shift alone would.

In [ ]:
#(t): The final WINDOW years of every member's regional means (regional_season_ds: A5)
final_regional_da = regional_season_ds[VARIABLE.name].isel(year=slice(-WINDOW, None))

In [ ]:
#(t): Each model's hist-nat mean over the same years: what its anomalies are measured from
hist_nat_final_mean_da = final_regional_da.sel(experiment='hist-nat').mean(['member', 'year'])

In [ ]:
#(t): Every member-year as an anomaly from its model's hist-nat
final_anomaly_da = final_regional_da - hist_nat_final_mean_da

In [ ]:
#(t): The x axis of the densities: a little beyond the smallest and largest anomalies (0.1 and 99.9 percentiles)
ridge_low, ridge_high = np.nanpercentile(final_anomaly_da.values, [0.1, 99.9])
ridge_x_arr = np.linspace(np.floor(ridge_low) - 1, np.ceil(ridge_high) + 1, 241)
ridge_x_arr[[0, -1]]

In [ ]:
#(t): The density of each experiment, every member of every model pooled, on (experiment, season, region, x)
pooled_density_da = fast_kde_xr(final_anomaly_da, ridge_x_arr, dim=['model', 'member', 'year'],
                                bandwidth=RIDGE_BANDWIDTH)

In [ ]:
#(t): The mean of each: its colour, and the number beside it
pooled_mean_da = final_anomaly_da.mean(['model', 'member', 'year'])
pooled_mean_da.sel(region=REGION).to_pandas().round(2)

In [ ]:
#(t): hist-nat's CHANCE and 1 - CHANCE quantiles (its 5th and 95th percentiles), every member of every model
#(t): pooled: the dotted lines
pooled_levels_da = nan_quantile(final_anomaly_da.sel(experiment='hist-nat'), [CHANCE, 1 - CHANCE],
                                ['model', 'member', 'year'])
pooled_levels_da.sel(region=REGION).to_pandas().round(2)

**R1. How each forcing changes the whole distribution.** Distributions of seasonal mean temperature over the region, 1993–2013, for hist-nat and each forcing (ridges), as anomalies from each model's hist-nat mean over the same years, every member of every model pooled; one panel per season. Colour and number: each distribution's mean. Dotted: hist-nat's 5th and 95th percentiles. One figure per region.

In [ ]:
#(t): R1, one figure per region: hist-nat and each forcing (ridges, top to bottom)
r1_figures = {}
with plt.rc_context(STYLE.rc):
    for region in oc.REGIONS:
        r1_figures[region] = plt.figure(figsize=(WIDTH, 3.6))
        draw_ridgelines(r1_figures[region], pooled_density_da.sel(region=region, experiment=['hist-nat',
                                                                                             *FORCED_EXPERIMENTS]),
                        pooled_mean_da.sel(region=region, experiment=['hist-nat', *FORCED_EXPERIMENTS]), 'experiment',
                        pooled_levels_da.sel(region=region))

In [ ]:
for region, fig in r1_figures.items():
    fr.save(fig, CANDIDATE_DIR, f"R1_ridgelines_{region.split(' (')[0].replace(' ', '_')}")

**R2. Every model.** As R1 for historical, every model on its own (each from its own hist-nat), with every model pooled at the top: how far the models agree on the shift and on the width.

In [ ]:
#(t): historical's anomalies
historical_anomaly_da = final_anomaly_da.sel(experiment='historical', drop=True)

In [ ]:
#(t): Each model's own density, on (model, season, region, x)
model_density_da = fast_kde_xr(historical_anomaly_da, ridge_x_arr, dim=['member', 'year'], bandwidth=RIDGE_BANDWIDTH)

In [ ]:
#(t): With every model pooled (R1's historical ridge) first, as 'All models'
all_models_density_da = pooled_density_da.sel(experiment='historical', drop=True).expand_dims(model=['All models'])
historical_density_da = xr.concat([all_models_density_da, model_density_da], dim='model')
historical_density_da

In [ ]:
#(t): The means, in the same order
all_models_mean_da = pooled_mean_da.sel(experiment='historical', drop=True).expand_dims(model=['All models'])
historical_mean_da = xr.concat([all_models_mean_da, historical_anomaly_da.mean(['member', 'year'])], dim='model')
historical_mean_da.sel(region=REGION).to_pandas().round(2)

In [ ]:
#(t): R2 alone: every model pooled, then each model (ridges, top to bottom)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 0.3 * historical_density_da.sizes['model'] + 2.3))
    axes, _ = draw_ridgelines(fig, historical_density_da.sel(region=REGION), historical_mean_da.sel(region=REGION),
                              'model', pooled_levels_da.sel(region=REGION), left_margin=1.1)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'R2_ridgelines_every_model_historical')

## D. Sea Ice and the Width

The expectation: where sea ice is lost, the open ocean holds the surface temperature close to freezing in winter less often, so the cold tail warms most and the distribution narrows; strongest in JJA and SON, weak in DJF. **D1** is B1 with the sea-ice change drawn (run *With sea ice*). D2 and D4 need the sea ice (JASMIN); D3 does not.

**D3. The change at every percentile in the sea-ice region.** As B3, for `ICE_REGION`: expected to fall to the right (the cold tail warming most) where ice is lost.

In [ ]:
#(t): D3 alone: the sea-ice region
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 3.0))
    axes, _ = draw_quantile_curves(fig, regional_quantile_curve_da, [ICE_REGION])

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'D3_quantile_curves_sea_ice_region')

### D2 and D4

**D2. Sea-ice loss against the change in width.** Each point is one model (marker) and forcing (colour): the change in sea-ice concentration against the change in width, both averaged over `ICE_REGION`, one panel per season; r is the correlation over all the points. **D4** is the same against the change in the mean, as a cross-check.

In [ ]:
#(t): The change in sea-ice concentration averaged over the region, on (model, experiment, season)
ice_regional_change_da = oc.regional_mean(ice_change_da, ICE_REGION)
ice_regional_change_da.sel(experiment=FORCED_EXPERIMENTS).to_dataframe().unstack('season').round(1)

In [ ]:
#(t): The change in width over the same region
width_ice_region_da = oc.regional_mean(qrange_change_ds['qrange_change'], ICE_REGION)
width_ice_region_da

In [ ]:
#(t): and in the mean
mean_ice_region_da = oc.regional_mean(mean_bootstrap_ds['mean_change'], ICE_REGION)
mean_ice_region_da

In [ ]:
#(t): D2 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 2.9))
    axes, _ = draw_scatter(fig, ice_regional_change_da, width_ice_region_da, 'Δ sea ice (%)',
                           f'Δ(Q95 − Q05) ({VARIABLE.units})')

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'D2_sea_ice_vs_width')

In [ ]:
#(t): D4 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 2.9))
    axes, _ = draw_scatter(fig, ice_regional_change_da, mean_ice_region_da, 'Δ sea ice (%)',
                           f'Δ mean ({VARIABLE.units})')

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'D4_sea_ice_vs_mean')

## E. SAM

From notebook calc_02: *The SAM phases*, and the *By SAM phase* subsections of sections 2 and 4. The phases are cut at hist-nat's terciles of the index, so hist-nat has a third of its seasons in each. On the sample the index is made up, so these only show that the figures work.

**E1. The forced change in the SAM.** Each forcing's change in the SAM index, 1993–2013 against hist-nat: the multi-model median (bars) and every model (markers).

In [ ]:
sam_change_da.to_dataframe().unstack('season').round(2)

In [ ]:
#(t): E1 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 2.9))
    axes, _ = draw_forcing_bars(fig, {'Δ SAM index': sam_change_da})

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'E1_sam_change')

**E2. The change in the mean in each SAM phase.** For each forcing (one figure each), the multi-model median change in the mean in the seasons of each SAM phase (rows), 1993–2013 against hist-nat's seasons in the same phase. Hatching as in A1.

In [ ]:
#(t): The AR6 agreement of the change in the mean in each phase
phase_mean_agreement_ds = sig.model_agreement(sam_mean_bootstrap_ds['mean_change'], sam_mean_bootstrap_ds['pvalue'] < ALPHA,
                                              sign_threshold=SIGN_THRESHOLD, significant_threshold=SIGNIFICANT_THRESHOLD)
phase_mean_agreement_ds

In [ ]:
#(t): E2, one figure per forcing: the models' median change in the mean in each SAM phase (rows), hatched where they
#(t): don't agree
e2_figures = {}
for experiment in FORCED_EXPERIMENTS:
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(PHASE_LABELS), n_cols=len(SEASONS),
                                                            n_colorbar_rows=1, n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes, next_tag = draw_agreement_maps(fig, phase_mean_agreement_ds.sel(experiment=experiment, phase=list(PHASE_LABELS),
                                                                              season=SEASONS),
                                             'phase', scales.change.colours(),
                                             row_labels=[f'{experiment}\n{label}' for label in PHASE_LABELS.values()])
    e2_figures[experiment] = fig

In [ ]:
for experiment, fig in e2_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'E2_sam_phase_mean_{experiment}')

**E3. The change in the SAM against the change in the mean** over the region, a point per model and forcing. **E5** is the same against the change in width.

In [ ]:
#(t): The change in the mean and in the width averaged over REGION, on (model, experiment, season)
mean_region_da = oc.regional_mean(mean_bootstrap_ds['mean_change'], REGION)
width_region_da = oc.regional_mean(qrange_change_ds['qrange_change'], REGION)
width_region_da

In [ ]:
#(t): E3 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 2.9))
    axes, _ = draw_scatter(fig, sam_change_da, mean_region_da, 'Δ SAM index', f'Δ mean ({VARIABLE.units})')

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'E3_sam_vs_mean')

**E4. The width in each SAM phase.** The change in width Q95 − Q05 of the internal variability over the region, in the seasons of each SAM phase (rows), 1993–2013 against hist-nat's seasons in the same phase: the multi-model median (bars) and every model (markers).

In [ ]:
#(t): The change in width in each phase, averaged over REGION
phase_region_da = oc.regional_mean(sam_qrange_change_ds['qrange_change'], REGION)
phase_region_da

In [ ]:
#(t): E4 alone: the change in width in each SAM phase (rows)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 1.6 * len(PHASE_LABELS) + 1.3))
    axes, _ = draw_forcing_bars(fig,
                                {f'Δ width, {PHASE_LABELS[phase]} ({VARIABLE.units})': phase_region_da.sel(phase=phase)
                                      for phase in PHASE_LABELS})

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'E4_width_by_sam_phase')

In [ ]:
#(t): E5 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 2.9))
    axes, _ = draw_scatter(fig, sam_change_da, width_region_da, 'Δ SAM index', f'Δ(Q95 − Q05) ({VARIABLE.units})')

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'E5_sam_vs_width')

## F. Synthesis and Additivity

**F1. Every forcing's change, and their sum.** Each experiment's change from its own 1850–1900 baseline, averaged over the region (notebook calc_02, sections 6 and 11): the mean (top) and the variance of the internal variability (bottom; variances add up where widths do not). The sum of the four single forcings stands next to historical: where the two differ, the forcings do not add up.

In [ ]:
#(t): Additivity of the mean (notebook calc_02, section 6) and of the variance (section 12)
mean_additivity_ds = rc.additivity(own_changes_da)
variance_additivity_ds = rc.additivity(own_variance_changes_da)

In [ ]:
#(t): Their terms averaged over REGION, on (model, term, season)
mean_terms_region_da = oc.regional_mean(mean_additivity_ds['response'], REGION)
variance_terms_region_da = oc.regional_mean(variance_additivity_ds['response'], REGION)
variance_terms_region_da

In [ ]:
#(t): F1 alone: each single forcing, their sum and historical (bars), the change in the mean (top) and in the variance
F1_BARS = [*rc.ADDITIVE_PARTS, 'sum of parts', 'historical']
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 4.5))
    axes, _ = draw_forcing_bars(fig, {f'Δ mean ({VARIABLE.units})': mean_terms_region_da,
                                      f'Δ variance ({VARIABLE.units}²)': variance_terms_region_da},
                                x_dim='term', bars=F1_BARS)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'F1_synthesis')

**F2. Where the forcings do not add up.** Multi-model median of historical's change minus the sum of the single forcings' changes, each from its own 1850–1900 baseline: the mean (top) and the variance of the internal variability (bottom). The residual mixes interaction between forcings, forcings left out (land use) and sampling noise.

In [ ]:
#(t): F2: the models' median residual of the additivity, of the mean (top) and of the variance (bottom)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(SEASONS), n_colorbar_rows=2,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(SEASONS))
    #(c): Each row's colour bar under it
    mean_cax = antarctica.colorbar_axis_below(axes[0])
    variance_cax = antarctica.colorbar_axis_below(axes[1])
    antarctica.heatmaps_of_antarctica(mean_additivity_ds['response'].sel(term='residual', season=SEASONS).median('model'), None,
                                      'season', axes=axes[0], cax=mean_cax, **scales.change.colours())
    antarctica.heatmaps_of_antarctica(variance_additivity_ds['response'].sel(term='residual', season=SEASONS).median('model'),
                                      None, 'season', axes=axes[1], cax=variance_cax, **variance_scale.colours())
    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['Mean', 'Variance'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'F2_additivity_residual')

## V. Verification Against ERA5

The second link of the attribution chain: historical has to be consistent with ERA5 for the models' forced changes to be credible for the real world (Suarez-Gutierrez et al., 2021). Regional means come first, as the paper recommends, with the grid-point maps as the spatial detail. From notebooks calc_03a and calc_03b (sections 5–9) and calc_03c.

**V1. Rank histograms.** ERA5's rank among each model's historical members (rows), regional means, 1979–2013, per season (columns), on ten bins of scaled rank (0: below every member, 1: above every member). Grey: the range a perfect model gives, each member in turn ranked against the others (5–95%). Flat bars inside the grey: ERA5 could be a member. A U shape: too little spread; a dome: too much; a slope: a different forced response. Corner: ERA5's percentile for the dispersion among the perfect-model values (✗ beyond their 5–95%). One figure each for historical's anomalies (signal and noise), historical's detrended anomalies (noise only) and hist-nat's anomalies (which ERA5 should *not* fit).

In [ ]:
#(t): V1, one figure per case: historical's anomalies, historical's detrended anomalies, hist-nat's anomalies
V1_CASES = {'historical_anomaly': ('historical', 'anomaly'), 'historical_detrended': ('historical', 'detrended'),
            'hist-nat_anomaly': ('hist-nat', 'anomaly')}
v1_figures = {}
with plt.rc_context(STYLE.rc):
    for name, (experiment, treatment) in V1_CASES.items():
        case_ranks_ds = regional_ranks_ds.sel(region=REGION, experiment=experiment, treatment=treatment)
        v1_figures[name] = plt.figure(figsize=(WIDTH, 1.1 * case_ranks_ds.sizes['model'] + 1.6))
        draw_rank_histograms(v1_figures[name], case_ranks_ds)

In [ ]:
for name, fig in v1_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'V1_rank_histograms_{name}')

**V2. Where each model is adequate.** (Top) How many models are adequate at each grid point (the paper's Fig. 8): ERA5 below every member in fewer than 10% of years, above every member in fewer than 10%, and inside the members' central 75% in at most 80% of them. (Below) Each model: where ERA5 is beyond every member too often, above (red) or below (blue), and hatched where it is in the central 75% too often (too much variability).

In [ ]:
#(t): How many models are adequate at each grid point
adequate_count_ds = ev.adequate_count(spatial_table_ds)
adequate_count_ds

In [ ]:
#(t): Each model's edge class (evp.EDGE_CLASSES)
edge_class_da = evp.edge_class(spatial_table_ds)
edge_class_da

In [ ]:
#(t): V2: how many models are adequate (top), and each model's edge class (rows), hatched where ERA5 is in its
#(t): members' central 75% in more than 80% of years
#(c): The models evaluated against ERA5 (notebook calc_03b)
n_eval_models = spatial_table_ds.sizes['model']
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=1 + n_eval_models, n_cols=len(SEASONS),
                                                        n_colorbar_rows=2, n_legends=1,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=1 + n_eval_models, n_cols=len(SEASONS))
    #(c): The count's colour bar under row 1, the classes' under the models' rows
    adequate_cax = antarctica.colorbar_axis_below(axes[0])
    edge_cax = antarctica.colorbar_axis_below(axes[1:])
    antarctica.heatmaps_of_antarctica(adequate_count_ds['n_adequate'].sel(season=SEASONS), None, 'season', axes=axes[0],
                                      cax=adequate_cax, **fr.majority_scale(n_eval_models, 'Adequate models').colours())
    antarctica.heatmaps_of_antarctica(edge_class_da.sel(season=SEASONS), 'model', 'season', axes=axes[1:], cax=edge_cax,
                                      **pp.class_scale(evp.EDGE_CLASSES, evp.EDGE_COLORS,
                                                       '% of years ERA5 is beyond every member', wrap=8).colours())
    antarctica.hatching_grid((spatial_table_ds['central_flag'] == 1).sel(model=edge_class_da['model'], season=SEASONS),
                             'model', 'season', axes=axes[1:], pattern='////')
    #(c): The legend under the colour bars
    central_handle = Patch(facecolor='white', edgecolor='0.35', hatch='////',
                           label="ERA5 in the members' central 75% in more than 80% of years")
    fig.legend(handles=[central_handle], loc='outside lower center', frameon=False, fontsize=STYLE.cbar_label_size)
    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['Adequate\nmodels', *edge_class_da['model'].values], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'V2_spatial_evaluation')

**V3. What is wrong, and is it the forcing or the variability?** Each model's diagnosis over 1979–2013 (left of the colour bar: adequate) and, for each problem, whether it is the same in 1979–1996 and 1997–2013 (the variability) or changes between them (the forced response).

In [ ]:
#(t): V3: each model's diagnosis (top rows) and its early-against-late diagnosis (bottom rows)
#(c): The models evaluated against ERA5 (notebook calc_03b); colour-bar rows of 1 in, for the classes' names on three lines
models = spatial_table_ds['model'].values
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2 * len(models), n_cols=len(SEASONS), n_colorbar_rows=2,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT,
                                                        colorbar_height=1.0), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2 * len(models), n_cols=len(SEASONS))
    #(c): The diagnoses' colour bar under their rows, early against late's under theirs
    diagnosis_cax = antarctica.colorbar_axis_below(axes[:len(models)])
    period_cax = antarctica.colorbar_axis_below(axes[len(models):])
    antarctica.heatmaps_of_antarctica(spatial_table_ds['diagnosis'].sel(season=SEASONS), 'model', 'season',
                                      axes=axes[:len(models)], cax=diagnosis_cax,
                                      **pp.class_scale(list(ev.DIAGNOSES.values()), evp.DIAGNOSIS_COLORS,
                                                       'Diagnosis', wrap=10).colours())
    antarctica.heatmaps_of_antarctica(spatial_table_ds['period_diagnosis'].sel(season=SEASONS), 'model', 'season',
                                      axes=axes[len(models):], cax=period_cax,
                                      **pp.class_scale(list(ev.PERIOD_DIAGNOSES.values()), evp.PERIOD_COLORS,
                                                       'Early against late').colours())
    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, [*models, *[f'{model}\nearly v late' for model in models]], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'V3_diagnosis')

**V4. Which experiments is the observed trend consistent with?** ERA5's 1979–2013 regional trend as a percentile of each experiment's members (columns), every model (rows) and season (panels). Outlined: outside the members' 5–95%. The attribution needs ERA5 inside historical and outside hist-nat.

In [ ]:
trend_consistency_ds = regional_consistency_ds.sel(region=REGION, statistic='trend')
trend_consistency_ds['percentile'].to_dataframe().unstack('season').round(0)

In [ ]:
#(t): V4 alone
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 0.22 * trend_consistency_ds.sizes['model'] + 2.6))
    axes, _ = draw_percentile_tables(fig, trend_consistency_ds['percentile'], trend_consistency_ds['verdict'],
                                     ['hist-nat', *FORCED_EXPERIMENTS])

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'V4_trend_consistency')

**V5. The moment test over the whole map** (notebook calc_03a, section 7), for the supplement as a table: ✗ where ERA5 is flagged over more of the map than a perfect model is 90% of the time.

In [ ]:
field_verdict_df = eval_map_table_ds['field_verdict'].sel(season=SEASONS).to_series().unstack(['statistic', 'season'])
field_verdict_df.replace({0.0: '✓', 1.0: '✗'})

## S. Supplementary

**S1. How often each SAM phase.** The share of each forcing's seasons in the negative (top) and the positive SAM phase (bottom), 1993–2013: a third in hist-nat, by construction. The multi-model median (bars) and every model (markers).

In [ ]:
#(t): S1 alone: the share of each forcing's seasons in the negative (top) and positive SAM phase
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 4.5))
    axes, _ = draw_forcing_bars(fig, {f'Share of seasons, {PHASE_LABELS[phase]}': sam_phase_share_da.sel(phase=phase)
                                      for phase in ('negative', 'positive')})

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'S1_sam_phase_share')

**S2. Every model's probability ratios.** As C3 for historical, every model (rows); dots: the 95% bootstrap range includes 1.

In [ ]:
#(t): historical's ratios in every model, and where they are not significant
model_ratio_da = log_ratio_da.sel(experiment='historical')
model_ratio_not_significant_da = ~pr_ds['pr_significant'].sel(experiment='historical', chance=CHANCE) \
    & model_ratio_da.notnull()
model_ratio_not_significant_da

In [ ]:
#(t): S2, one figure per tail: historical's probability ratio in every model (rows), dotted where not significant
s2_figures = {}
models = model_ratio_da['model'].values
for tail in ('warm', 'cold'):
    with plt.rc_context(STYLE.rc):
        fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                            n_legends=1,
                                                            panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                         layout='constrained')
        axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
        cax = antarctica.colorbar_axis_below(axes)
        antarctica.heatmaps_of_antarctica(model_ratio_da.sel(tail=tail, season=SEASONS), 'model', 'season', axes=axes,
                                          cax=cax, **ratio_scale.colours())
        antarctica.not_significant_dots_grid(model_ratio_not_significant_da.sel(tail=tail, season=SEASONS), 'model', 'season',
                                             axes=axes)
        antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
        antarctica.label_rows(axes, models, fontsize=STYLE.label_size)
        antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
        fig.legend(handles=[not_significant_handle], loc='outside lower center', fontsize=STYLE.cbar_label_size)
    s2_figures[tail] = fig

In [ ]:
for tail, fig in s2_figures.items():
    fr.save(fig, CANDIDATE_DIR, f'S2_probability_ratios_every_model_{tail}')

## Candidate Layouts

The blocks above, put together (`fig.subfigures`, one part per block), the panel letters running on from block to block. A layout figure that is a single candidate is not drawn again:

| Layout | Fig. 1 | Fig. 2 | Fig. 3 | Fig. 4 | Fig. 5 |
|---|---|---|---|---|---|
| L1 current plan | A1 + C1 | B1 + C2 | D2 | E2 + E3 | |
| L2 merged mechanism | A1 + C1 | B1 + C2 | D2, D3 / E1, E2 | F1 | |
| L3 original outline | A2 (historical) | the width as A2 | A1 | B1 | F2 |
| L4 obs-led | A4 + A5 | A1 | B1 or B3 | F1 | D2 + E2 |
| L5 short NCC | the strongest result (a candidate: R1 + C2) | A1 or F1 | D2 + E3 | C2 | |

The figures with D2 need the sea ice, so they are in the (JASMIN) section.

**L1 and L2, Fig. 1: the mean change and its extremes.** A1 above C1.

In [ ]:
#(t): L1, figure 1: A1's maps above C1's probability ratios
with plt.rc_context(STYLE.rc):
    fig = plt.figure(layout='constrained', figsize=(WIDTH, 6.6 + 4.2))
    maps_fig, ratios_fig = fig.subfigures(2, 1, height_ratios=[6.6, 4.2])
    axes, next_tag = draw_agreement_maps(maps_fig, mean_agreement_ds.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS),
                                         'experiment', scales.change.colours(), n_models_da=mean_n_models_da)
    draw_probability_ratios(ratios_fig, regional_ratio_ds, tag_start=next_tag)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L1_fig1_mean_and_extremes')

**L1 and L2, Fig. 2: the width and what it does to the extremes.** B1 above C2.

In [ ]:
#(t): L1, figure 2: B1's maps above C2's ratios and shift-only ratios
with plt.rc_context(STYLE.rc):
    fig = plt.figure(layout='constrained', figsize=(WIDTH, 6.6 + 4.2))
    maps_fig, ratios_fig = fig.subfigures(2, 1, height_ratios=[6.6, 4.2])
    axes, next_tag = draw_agreement_maps(maps_fig, width_agreement_ds.sel(experiment=FORCED_EXPERIMENTS,
                                                                          season=SEASONS),
                                         'experiment', scales.spread_change.colours(), n_models_da=width_n_models_da)
    draw_probability_ratios(ratios_fig, regional_both_ratios_ds, shift_only=True, tag_start=next_tag)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L1_fig2_width_and_extremes')

**L1, Fig. 4: the SAM.** E2 for historical above E3.

In [ ]:
#(t): L1, figure 4: E2's maps for historical above E3's scatter
with plt.rc_context(STYLE.rc):
    fig = plt.figure(layout='constrained', figsize=(WIDTH, 5.0 + 2.9))
    maps_fig, scatter_fig = fig.subfigures(2, 1, height_ratios=[5.0, 2.9])
    axes, next_tag = draw_agreement_maps(maps_fig, phase_mean_agreement_ds.sel(experiment='historical',
                                                                               phase=list(PHASE_LABELS), season=SEASONS),
                                         'phase', scales.change.colours(),
                                         row_labels=[f'historical\n{label}' for label in PHASE_LABELS.values()])
    draw_scatter(scatter_fig, sam_change_da, mean_region_da, 'Δ SAM index', f'Δ mean ({VARIABLE.units})',
                 tag_start=next_tag)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L1_fig4_sam')

**L3, Fig. 2: the width, as A2.** For historical: the median change in width, and how many models agree on its sign.

In [ ]:
#(t): How many models agree on the sign of the median change in width
width_sign_count_da = (width_agreement_ds['sign_agreement'] * width_agreement_ds['n_models']).round()
width_sign_count_da

In [ ]:
#(t): L3, figure 2: historical's median change in the width (top) and how many models agree on its sign (bottom)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(SEASONS), n_colorbar_rows=2,
                                                        panel_size=MAP_PANEL_SIZE, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(SEASONS))
    #(c): Each row's colour bar under it
    median_cax = antarctica.colorbar_axis_below(axes[0])
    count_cax = antarctica.colorbar_axis_below(axes[1])
    antarctica.heatmaps_of_antarctica(width_agreement_ds['median'].sel(experiment='historical', season=SEASONS), None, 'season',
                                      axes=axes[0], cax=median_cax, **scales.spread_change.colours())
    antarctica.heatmaps_of_antarctica(width_sign_count_da.sel(experiment='historical', season=SEASONS), None, 'season',
                                      axes=axes[1], cax=count_cax, **count_scale.colours())
    antarctica.label_columns(axes, SEASONS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['historical\nmedian', 'Models\nagreeing'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L3_fig2_width_historical')

**L4, Fig. 1: the observed change.** A4 above A5.

In [ ]:
#(t): L4, figure 1: A4's maps above A5's time series
with plt.rc_context(STYLE.rc):
    fig = plt.figure(layout='constrained', figsize=(WIDTH, 7.0 + 2.6))
    maps_fig, series_fig = fig.subfigures(2, 1, height_ratios=[7.0, 2.6])
    axes, next_tag = draw_observed_maps(maps_fig)
    draw_plumes(series_fig, plume_da.sel(region=REGION), era5_anomaly_da.sel(region=REGION), tag_start=next_tag)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L4_fig1_observed')

**L5, Fig. 1: a candidate headline.** The whole distributions over the region (R1) above what that does to the extremes (C2).

In [ ]:
#(t): L5, figure 1: R1 for REGION above C2
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 3.6 + 4.2))
    ridges_fig, ratios_fig = fig.subfigures(2, 1, height_ratios=[3.6, 4.2])
    axes, next_tag = draw_ridgelines(ridges_fig,
                                     pooled_density_da.sel(region=REGION, experiment=['hist-nat', *FORCED_EXPERIMENTS]),
                                     pooled_mean_da.sel(region=REGION, experiment=['hist-nat', *FORCED_EXPERIMENTS]),
                                     'experiment', pooled_levels_da.sel(region=REGION))
    draw_probability_ratios(ratios_fig, regional_both_ratios_ds, shift_only=True, tag_start=next_tag)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L5_fig1_distributions_and_extremes')

### Layouts with sea ice

**L2, Fig. 3: the mechanisms.** The sea-ice row (D2, D3) above the SAM row (E1, E2 for historical).

In [ ]:
#(t): L2, figure 3: D2's scatter, D3's curves, E1's bars and E2's maps for historical
with plt.rc_context(STYLE.rc):
    fig = plt.figure(layout='constrained', figsize=(WIDTH, 2.9 + 3.0 + 2.9 + 5.0))
    scatter_fig, curves_fig, bars_fig, maps_fig = fig.subfigures(4, 1, height_ratios=[2.9, 3.0, 2.9, 5.0])
    axes, next_tag = draw_scatter(scatter_fig, ice_regional_change_da, width_ice_region_da, 'Δ sea ice (%)',
                                  f'Δ(Q95 − Q05) ({VARIABLE.units})')
    axes, next_tag = draw_quantile_curves(curves_fig, regional_quantile_curve_da, [ICE_REGION], tag_start=next_tag)
    axes, next_tag = draw_forcing_bars(bars_fig, {'Δ SAM index': sam_change_da}, tag_start=next_tag)
    axes, next_tag = draw_agreement_maps(maps_fig, phase_mean_agreement_ds.sel(experiment='historical',
                                                                               phase=list(PHASE_LABELS), season=SEASONS),
                                         'phase', scales.change.colours(),
                                         row_labels=[f'historical\n{label}' for label in PHASE_LABELS.values()],
                                         tag_start=next_tag)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L2_fig3_mechanisms')

**L4, Fig. 5: the mechanisms, short.** D2 above E2 for historical.

In [ ]:
#(t): L4, figure 5: D2's scatter above E2's maps for historical
with plt.rc_context(STYLE.rc):
    fig = plt.figure(layout='constrained', figsize=(WIDTH, 2.9 + 5.0))
    scatter_fig, maps_fig = fig.subfigures(2, 1, height_ratios=[2.9, 5.0])
    axes, next_tag = draw_scatter(scatter_fig, ice_regional_change_da, width_ice_region_da, 'Δ sea ice (%)',
                                  f'Δ(Q95 − Q05) ({VARIABLE.units})')
    axes, next_tag = draw_agreement_maps(maps_fig, phase_mean_agreement_ds.sel(experiment='historical',
                                                                               phase=list(PHASE_LABELS), season=SEASONS),
                                         'phase', scales.change.colours(),
                                         row_labels=[f'historical\n{label}' for label in PHASE_LABELS.values()],
                                         tag_start=next_tag)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L4_fig5_mechanisms')

**L5, Fig. 3: the mechanisms, as scatter.** D2 above E3.

In [ ]:
#(t): L5, figure 3: D2 above E3
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=(WIDTH, 2.9 + 2.9))
    d2_fig, e3_fig = fig.subfigures(2, 1, height_ratios=[2.9, 2.9])
    axes, next_tag = draw_scatter(d2_fig, ice_regional_change_da, width_ice_region_da, 'Δ sea ice (%)',
                                  f'Δ(Q95 − Q05) ({VARIABLE.units})')
    draw_scatter(e3_fig, sam_change_da, mean_region_da, 'Δ SAM index', f'Δ mean ({VARIABLE.units})', tag_start=next_tag)

In [ ]:
fr.save(fig, CANDIDATE_DIR, 'L5_fig3_mechanisms')

## What Was Saved

In [ ]:
pd.Series(sorted(path.name for path in CANDIDATE_DIR.glob('*.pdf')), name=str(CANDIDATE_DIR)).to_frame()